In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl

# Optional für Plots
import matplotlib.pyplot as plt

In [16]:


# Excel-Datei im selben Ordner wie das Notebook:
xlsx_path = Path.cwd() / "AMLTA Experiment.xlsx"
xls = pd.ExcelFile(xlsx_path)
xls.sheet_names
sheet = "Experiment Data"

In [17]:
df_races = pd.read_excel(
    xlsx_path,
    sheet_name=sheet,
    header=4,          # Zeile 5 enthält Header
    usecols="E:K",     # nur diese Spalten
    nrows=64           # 68 - 5 + 1 = 64 Zeilen
)

df_laps = pd.read_excel(
    xlsx_path,
    sheet_name=sheet,
    header=4,
    usecols="N:U",
    nrows=316          # 320 - 5 + 1 = 316 Zeilen
)

df_races.head()

,Name.1,ID.1,Strecke,Hilfe,Erfahrung.1,Unnamed: 9,Gesamtdauer
0,AmL ID5,1.0,1.0,1.0,1.0,NaN,313.02
1,AmL ID5,1.0,2.0,2.0,1.0,NaN,510.20
2,AmL ID5,1.0,3.0,3.0,1.0,NaN,497.33
3,ML ID7,2.0,2.0,2.0,1.0,NaN,464.05
4,ML ID7,2.0,3.0,3.0,1.0,NaN,476.25


In [18]:
df_races = df_races.dropna(how="all")
df_laps  = df_laps.dropna(how="all")


In [19]:
df_races["Gesamtdauer"] = (
    df_races["Gesamtdauer"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [20]:
# save to csv
df_races.to_csv("races.csv", index=False)
df_laps.to_csv("laps.csv", index=False) 

In [22]:
hilfe_map = {
    1: "Ohne Hilfe",
    2: "Mit Coach",
    3: "Mit Fahrlinie"
}

df_races["Hilfe_label"] = df_races["Hilfe"].map(hilfe_map)
df_laps["Hilfe_label"] = df_laps["Hilfe.1"].map(hilfe_map)


In [23]:
grouped = (
    df_races
    .groupby(["Hilfe_label", "Strecke"])["Gesamtdauer"]
    .mean()
    .reset_index()
)

grouped

,Hilfe_label,Strecke,Gesamtdauer
0,Mit Coach,1.0,106.588571
1,Mit Coach,2.0,230.860000
2,Mit Coach,3.0,121.081667
3,Mit Fahrlinie,1.0,89.205000
4,Mit Fahrlinie,2.0,116.134286
5,Mit Fahrlinie,3.0,229.691250
6,Ohne Hilfe,1.0,152.932500
7,Ohne Hilfe,2.0,117.533333
8,Ohne Hilfe,3.0,136.001429


In [24]:
mean_total = (
    df_races
    .groupby("Hilfe_label")["Gesamtdauer"]
    .mean()
    .reset_index()
)

plt.figure()
plt.bar(mean_total["Hilfe_label"], mean_total["Gesamtdauer"])
plt.xlabel("Hilfe")
plt.ylabel("Durchschnittliche Gesamtdauer (s)")
plt.title("Gesamtdauer nach Hilfe")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("gesamtdauer_nach_hilfe.png")
plt.close()

In [25]:
mean_lap = (
    df_laps
    .groupby(["Hilfe_label", "Runde"])["Dauer"]
    .mean()
    .reset_index()
)

plt.figure()

for hilfe in mean_lap["Hilfe_label"].unique():
    subset = mean_lap[mean_lap["Hilfe_label"] == hilfe]
    plt.plot(subset["Runde"], subset["Dauer"], marker="o", label=hilfe)

plt.xlabel("Runde")
plt.ylabel("Durchschnittliche Rundendauer (s)")
plt.title("Lap-to-Lap Entwicklung nach Hilfe")
plt.legend()
plt.tight_layout()
plt.savefig("lap_steigerung_nach_hilfe.png")
plt.close()